# Local-loopback requests

Install Deno and register its TypeScript kernel with `deno jupyter --install`. Open this file
in a Jupyter front end such as JupyterLab or VS Code, select **Deno**, then Run All in a fresh kernel.
The cells import exact published signer **2.0.0** and verifier **2.0.0**.

Use the explicit loopback profile for literal 127.0.0.1 or ::1 addresses. The profile requires
a server challenge and refuses hostnames and remote addresses. A real HTTP listener below checks
request binding. The host still owns authorization, trusted keys, revocation and replay.


## Load the published packages

Check both release identities and the backend Deno selects.


In [ ]:
import assert from "node:assert/strict";
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import * as signer from "npm:@bounded-authority-protocol/signer@2.0.0";
import * as verifier from "npm:@bounded-authority-protocol/verifier@2.0.0";
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
assert.equal(signerMetadata.version, "2.0.0");
assert.equal(verifierMetadata.version, "2.0.0");
assert.equal(verifier.BACKEND_ID, "portable");
console.log(JSON.stringify({ signer: signerMetadata.version, verifier: verifierMetadata.version, backend: verifier.BACKEND_ID }));


{"signer":"2.0.0","verifier":"2.0.0","backend":"portable"}


## Prepare real demonstration key handles

Each constructor holds a fresh private key in its closure and exposes only the key-handle contract.
No key material is displayed. Production custody supplies this same contract.


In [ ]:
function createEd25519Handle(keyId: string, role: signer.KeyRole = "holder"): signer.KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const publicBytes = new Uint8Array(Buffer.from(publicKey.export({ format: "jwk" }).x!, "base64url"));
  return {
    sign: (signingInput) => new Uint8Array(sign(null, signingInput, privateKey)),
    publicKey: () => publicBytes,
    thumbprint: () => verifier.thumbprint(verifier.jwkFromPublicKey(publicBytes)),
    keyIdentity: () => ({ keyId, publicKey: publicBytes }),
    signingIdentity: () => ({ role, keyId, publicKey: publicBytes }),
  };
}
function createEs256Handle(keyId: string, role: signer.KeyRole = "holder"): signer.Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const publicBytes = new Uint8Array([0x04, ...Buffer.from(x, "base64url"), ...Buffer.from(y, "base64url")]);
  return {
    sign: (signingInput) => new Uint8Array(sign("sha256", signingInput, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => publicBytes,
    thumbprint: () => verifier.v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: publicBytes }),
    signingIdentity: () => ({ role, keyId, publicKey: publicBytes }),
  };
}
const ed25519Width = (await createEd25519Handle("example-ed25519").publicKey()).length;
const es256Width = (await createEs256Handle("example-es256").publicKey()).length;
assert.equal(ed25519Width, 32);
assert.equal(es256Width, 65);
console.log(JSON.stringify({ ed25519Bytes: ed25519Width, es256Bytes: es256Width }));


{"ed25519Bytes":32,"es256Bytes":65}


## Issue the grant

The logical resource audience is separate from the literal loopback target.


In [ ]:
const issuer = createEd25519Handle("issuer-key", "issuer");
const holder = createEd25519Handle("holder-key");
const grant = await signer.signGrant({ issuer: "https://issuer.example", grantId: "loopback-grant",
  audiences: ["https://resource.example"], issuedAt: 1_000, notBefore: 1_000, expiresAt: 2_000,
  holderThumbprint: await holder.thumbprint(), operations: [{ name: "inspect", selectors: [{ kind: "all" }] }],
}, issuer);
assert.ok(grant.ok);
const request = { method: "POST", targetUri: "http://127.0.0.1:47321/inspect",
  invocationId: "550e8400-e29b-41d4-a716-446655440000", operation: "inspect", castArguments: { t: "null" } as const };
const nonce = "server-challenge";
console.log(JSON.stringify({ ok: grant.ok }));


{"ok":true}


## Sign the challenged loopback request

The holder signs the literal target and the server-issued nonce.


In [ ]:
const envelope = await signer.signLocalLoopbackReport({ ...request, grantCompact: grant.value.grant, nonce }, holder, { issuedAt: 1_100, proofId: "loopback-proof" });
assert.ok(envelope.ok);
console.log(JSON.stringify({ ok: envelope.ok }));


{"ok":true}


## Verify the loopback profile

Supply a required-nonce policy and already-trusted issuer key.


In [ ]:
const expected = { ...request, trustedIssuer: { keyId: "issuer-key", publicKey: await issuer.publicKey() },
  issuer: "https://issuer.example", audience: "https://resource.example", evaluationTime: 1_120,
  clockSkew: 0, proofMaxAge: 300, nonce: { kind: "required", value: nonce } as const };
const facts = verifier.checkLocalLoopbackHttpEnvelope(envelope.value.grant, envelope.value.proof, expected);
assert.ok(facts.ok);
console.log(JSON.stringify({ ok: facts.ok, operation: facts.value.operation, authorization: facts.value.authorization }));


{"ok":true,"operation":"inspect","authorization":"not_evaluated"}


## Refuse a missing challenge

The loopback signer requires a non-empty server nonce.


In [ ]:
const missingNonce = await signer.signLocalLoopbackReport({ ...request, grantCompact: grant.value.grant }, holder, { issuedAt: 1_100, proofId: "refusal-proof" });
assert.equal(missingNonce.ok, false);
if (missingNonce.ok) throw new Error("invalid loopback request accepted");
assert.equal(missingNonce.error, "invalid_report");
console.log(JSON.stringify({ ok: missingNonce.ok, code: missingNonce.error }));


{"ok":false,"code":"invalid_report"}


## Refuse a hostname

The profile accepts only literal loopback addresses.


In [ ]:
const hostname = await signer.signLocalLoopbackReport({ ...request, targetUri: "http://localhost:47321/inspect", grantCompact: grant.value.grant, nonce }, holder, { issuedAt: 1_100, proofId: "refusal-proof" });
assert.equal(hostname.ok, false);
if (hostname.ok) throw new Error("invalid loopback request accepted");
assert.equal(hostname.error, "producer_error");
console.log(JSON.stringify({ ok: hostname.ok, code: hostname.error }));


{"ok":false,"code":"producer_error"}


## Refuse a remote address

A literal address outside loopback does not qualify for this profile.


In [ ]:
const remote = await signer.signLocalLoopbackReport({ ...request, targetUri: "http://192.0.2.1:47321/inspect", grantCompact: grant.value.grant, nonce }, holder, { issuedAt: 1_100, proofId: "refusal-proof" });
assert.equal(remote.ok, false);
if (remote.ok) throw new Error("invalid loopback request accepted");
assert.equal(remote.error, "producer_error");
console.log(JSON.stringify({ ok: remote.ok, code: remote.error }));


{"ok":false,"code":"producer_error"}


## Start a real loopback endpoint

The server reconstructs context from the received HTTP method and target. It returns only the computed verification result.


In [ ]:
import { createServer } from "node:http";
import { once } from "node:events";
const server = createServer((received, response) => {
  const receivedGrant = verifier.strUtf8(String(received.headers["x-example-grant"] ?? ""));
  const receivedProof = verifier.strUtf8(String(received.headers["x-example-proof"] ?? ""));
  const checked = verifier.checkLocalLoopbackHttpEnvelope(receivedGrant, receivedProof, {
    ...expected, method: received.method!, targetUri: `http://127.0.0.1:${port}${received.url}`,
  });
  response.writeHead(checked.ok ? 200 : 401, { "content-type": "application/json" });
  response.end(JSON.stringify({ verified: checked.ok }));
});
server.listen(0, "127.0.0.1");
await once(server, "listening");
const address = server.address();
assert.ok(address && typeof address !== "string");
const port = address.port;
assert.equal(server.listening, true);
console.log(JSON.stringify({ listening: server.listening }));


{"listening":true}


## Send the exact signed target

Sign for the free port the real listener selected, then send those bytes.


In [ ]:
const targetUri = `http://127.0.0.1:${port}/inspect`;
const httpEnvelope = await signer.signLocalLoopbackReport({ ...request, targetUri, grantCompact: grant.value.grant, nonce }, holder, { issuedAt: 1_100, proofId: "http-proof" });
assert.ok(httpEnvelope.ok);
const headers = { "x-example-grant": verifier.utf8Str(httpEnvelope.value.grant), "x-example-proof": verifier.utf8Str(httpEnvelope.value.proof) };
const acceptedResponse = await fetch(targetUri, { method: "POST", headers });
const acceptedBody = await acceptedResponse.json() as { verified: boolean };
assert.equal(acceptedResponse.status, 200);
assert.equal(acceptedBody.verified, true);
console.log(JSON.stringify({ status: acceptedResponse.status, verified: acceptedBody.verified }));


{"status":200,"verified":true}


## Refuse a changed HTTP target

Reuse the same compact bytes against a different received target.


In [ ]:
const changedResponse = await fetch(targetUri + "-changed", { method: "POST", headers });
const changedBody = await changedResponse.json() as { verified: boolean };
assert.equal(changedResponse.status, 401);
assert.equal(changedBody.verified, false);
console.log(JSON.stringify({ status: changedResponse.status, verified: changedBody.verified }));


{"status":401,"verified":false}


## Close the real listener

Release the listener and all connections. Kernel shutdown also releases these resources if an earlier assertion fails.


In [ ]:
server.closeAllConnections();
await new Promise<void>((resolve, reject) => server.close((error) => error ? reject(error) : resolve()));
assert.equal(server.listening, false);
console.log(JSON.stringify({ listening: server.listening }));


{"listening":false}


## Finish

Shut down the kernel to discard demonstration keys. Continue with the [loopback guide](../guides/local-loopback.md).
